# Mira Unsloth Fine-tune on Colab Free
Base: Qwen/Qwen3-8B
Target: Hoodx/mira-agent-instinct

Runtime: Change runtime type -> T4 GPU

In [ ]:
!pip install -q unsloth bitsandbytes transformers accelerate datasets trl peft huggingface_hub

In [ ]:
from google.colab import files
uploaded = files.upload()
# Upload data/raw.jsonl

In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen3-8B",
    max_seq_length=1024,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(model, r=8, lora_alpha=8)
print("Model loaded")

In [ ]:
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

dataset = load_dataset("json", data_files="data/train.jsonl", split="train")

training_args = TrainingArguments(
    output_dir="mira-finetuned",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=10,
    save_steps=500,
    fp16=True,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=1024,
    args=training_args,
)
trainer.train()

In [ ]:
model.save_pretrained("mira-finetuned")
tokenizer.save_pretrained("mira-finetuned")
print("Saved to mira-finetuned")

In [ ]:
from huggingface_hub import login, HfApi
from huggingface_hub.utils import RepositoryNotFoundError

hf_token = ""  # paste your HF token
login(token=hf_token)

api = HfApi()
repo_id = "Hoodx/mira-agent-instinct"
api.create_repo(repo_id, exist_ok=True)

from huggingface_hub import upload_folder
upload_folder(folder_path="mira-finetuned", repo_id=repo_id, repo_type="model")
print(f"Pushed to {repo_id}")